In [12]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1



for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue # 논문제목 못찾으면 오류 발생, 해당 항목 건너뜀 
        else :
            f = open(ft_name, 'a' , encoding="UTF-8") #append(추가모드), 기존 파일에 계속 추가 
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )

            no += 1
            print("\n")

            
            if no > collect_cnt :
                break

            time.sleep(1) 

    a += 1
    b = str(a)
            

    time.sleep(1) # 페이지 변경 전 1초 대기

  
    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")
    time.sleep(1) # 페이지 변경 전 1초 대기



# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False,  engine='openpyxl')

#encoding="utf-8" ,

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
1해양관광유형과 관광자원 가치가 주민참여의도에 미치는 영향 : 당진시를 중심으로이길호경기대학교 관광전문대학원2014국내박사RANK : 13880191원문보기목차검색조회음성듣기21세기는 문화시대이자 창의적 산업시대이며 감성적 소통 시대이다. 최근 들어 국민소득 증대와 함께 생활수준이 향상되고 주 5일 근무 정착에 따른 여가시간 확대로 관광 및 레저에 대한 관심도가 증가하고 있으며 관광 패턴이 육지위주에서 해양위주로 변화하고 있다. 3면이 바다로 둘러 싸여있는 우리나라는 천혜의 해양관광자원 보고이며 축복받은 해양관광자원이 곳곳에 산재되어 있다. 따라서 해양관광자원을 개발위주의 근대 공간에서 탈피하고 장소성이 있는 공간으로 창출하여 관광자와 지역주민의 삶의 질을 높이는 한편, 자연과 환경을 보전하고 지역주민 경제적 활성화를 증대 시키는 일이 요구되고 있다. 따라서 지방자치제도 시행 이후 각 지자체들은 지역경제 활성화 방안으로 지역종합발전계획을 수립하고 더 많은 외래관광객 유입을 목표로 관광시책을 추진하고 있으나 지자체 관심과 의욕에 비하여 관광발전시책이 성공적이지 못한 사례가 여러 곳에서 나타나고 있는 현실이다. 그 원인의 대부분은 관광객 유인을 위한 방안으로 관주도의 일방적인 관광개발에 치중하여 새로운 관광지 조성을 위한 개발 사업에 주력해 왔기 때문이다. 이는 공급자 중심 시설개발에서 수요자 중심 관광만족도 제고를 위한 방법이 미흡한 것으로 해석된다. 따라서 관주도 관광자원 관리방식을 탈피하여 지자체와 관광전문가, 관광사업자, 그리고 지역주민과 협의 하에 보호, 보전, 개발의 관광자원 관리의 필요성이 중요시 되고 있다.우리나라는 육지면적의 3배가 넘는 34만 5,000㎢에 달하는 대륙붕과 수심 20m 내․외 해역도 국토 3분의 1에 해당하며, 11,542㎞에 달하는 해안선을 보유하고 있다. 또한 3,200여 개의 도서를 보유하고 있으며, 갯벌 면적은 남한 면적 2.5%에 달하는 2,815㎢

In [6]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
degree2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 + 논문집
            etc = cont.find('p','etc')
            spans = etc.find_all('span')

            year = spans[-2].get_text().strip()
            degree = spans[-1].get_text().strip()

            print('5.발행년도:',year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            print('6.논문집:',degree)
            degree2.append(degree)
            f.write('\n6.논문집:' + degree)

            # URL
            detail_url = "https://www.riss.kr" + title_tag['href']
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,'다음 페이지로').click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
25 건을 수집하기 위해 3 페이지를 조회합니다.
1.번호: 1
2.논문제목: 해양관광유형과 관광자원 가치가 주민참여의도에 미치는 영향 : 당진시를 중심으로
3.저자: 이길호
4.소속기관: 경기대학교 관광전문대학원
5.발행년도: 2014
6.논문집: 국내박사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=7c7ccbfcd70413ddffe0bdc3ef48d419&keyword=해양자원


1.번호: 2
2.논문제목: 해양경계획정 이전 권원중첩수역에서의 자원개발과 환경보호
3.저자: 이영주
4.소속기관: 연세대학교 대학원
5.발행년도: 2014
6.논문집: 국내석사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=0921670f9b2d4d76ffe0bdc3ef48d419&keyword=해양자원


1.번호: 3
2.논문제목: 국가관할권 이원지역 해양유전자원의 접근 및 이익공유에 관한 연구
3.저자: 조은영
4.소속기관: 중앙대학교 대학원
5.발행년도: 2019
6.논문집: 국내석사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=c528433398ce596affe0bdc3ef48d419&keyword=해양자원


1.번호: 4
2.논문제목: 島嶼觀光資源開發이 海洋觀光産業에 미치는 經濟的 波及效果
3.저자: 김두성
4.소속기관: 부산대학교
5.발행년도: 2012
6.논문집: 국내박사
7.URL: https://www.riss.kr/search/detail/DetailView.

In [ ]:
#!pip install openpyxl


In [ ]:
# find('a')['href']

In [7]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
degree2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 + 논문집
            etc = cont.find('p','etc')
            spans = etc.find_all('span')

            year = spans[-2].get_text().strip()
            degree = spans[-1].get_text().strip()

            print('5.발행년도:',year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            print('6.논문집:',degree)
            degree2.append(degree)
            f.write('\n6.논문집:' + degree)

            # URL
            detail_url = "https://www.riss.kr" + title_tag['href']
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
25 건을 수집하기 위해 3 페이지를 조회합니다.
1.번호: 1
2.논문제목: 해양관광유형과 관광자원 가치가 주민참여의도에 미치는 영향 : 당진시를 중심으로
3.저자: 이길호
4.소속기관: 경기대학교 관광전문대학원
5.발행년도: 2014
6.논문집: 국내박사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=7c7ccbfcd70413ddffe0bdc3ef48d419&keyword=해양자원


1.번호: 2
2.논문제목: 해양경계획정 이전 권원중첩수역에서의 자원개발과 환경보호
3.저자: 이영주
4.소속기관: 연세대학교 대학원
5.발행년도: 2014
6.논문집: 국내석사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=0921670f9b2d4d76ffe0bdc3ef48d419&keyword=해양자원


1.번호: 3
2.논문제목: 국가관할권 이원지역 해양유전자원의 접근 및 이익공유에 관한 연구
3.저자: 조은영
4.소속기관: 중앙대학교 대학원
5.발행년도: 2019
6.논문집: 국내석사
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=c528433398ce596affe0bdc3ef48d419&keyword=해양자원


1.번호: 4
2.논문제목: 島嶼觀光資源開發이 海洋觀光産業에 미치는 經濟的 波及效果
3.저자: 김두성
4.소속기관: 부산대학교
5.발행년도: 2012
6.논문집: 국내박사
7.URL: https://www.riss.kr/search/detail/DetailView.

국내학술논문에서 보는거구나 

1. 사용자에게 RISS 사이트의 모든 카테고리를 다 보여준 후 사용자에게 “국내학술논문” 카테고리
를 선택했다고 가정하여 수집할 장르를 2번으로 입력 후 저장할 파일의 이름을 입력 받은 후 “국
내학술논문” 카테고리의 다양한 정보들을 수집하여 csv, xls 형식으로 저장하도록 코드를 작성하세
요. 수집할 항목은 아래의 예시를 참고하세요.

In [14]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
degree2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 + 논문집
            etc = cont.find('p','etc')
            spans = etc.find_all('span')

            year = spans[-2].get_text().strip()
            degree = spans[-1].get_text().strip()

            print('5.발행년도:',year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            print('6.논문집:',degree)
            degree2.append(degree)
            f.write('\n6.논문집:' + degree)

            # URL
            detail_url = "https://www.riss.kr" + title_tag['href']
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
25 건을 수집하기 위해 3 페이지를 조회합니다.
1.번호: 1
2.논문제목: 발표논문 / 해양생물자원으로서 해조류 : 생물활성물질의 정제와 분자적 응용
3.저자: 홍용기(Yong Ki Hong)
4.소속기관: 한국조류학회
5.발행년도: 자원
6.논문집: Vol.- No.-
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=508c47ca1906d5ecffe0bdc3ef48d419&keyword=해양자원


1.번호: 2
2.논문제목: 수거된 해양폐기물 자원화 기술 개발(Ⅰ)
3.저자: 길상인(Sang-In Keel),윤진한(Jin-Han Yun),최연석(Yeon-Seok Choi),강창구(Chang-Gu Kang),유정석(Jeong-Seok Yu)
4.소속기관: 한국해양환경·에너지학회
5.발행년도: 해양
6.논문집: Vol.5 No.2
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=09ec1e3541fcc118ffe0bdc3ef48d419&keyword=해양자원


1.번호: 3
2.논문제목: 조건부 가치측정법을 이용한 공해상 해양생명자원 확보의 경제적 가치 추정
3.저자: 진세준,권영주,최은철
4.소속기관: 해양환경안전학회
5.발행년도: 해양
6.논문집: Vol.29 No.7
7.URL: https://www.riss.kr/search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=59e951e287ce111647de9c1710b0298d&keyword=해양자원


1.번호: 4
2.논문제목: 해양수산생명자원의 국내 접근 절차 

In [13]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)


            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
25 건을 수집하기 위해 3 페이지를 조회합니다.
1.번호: 1
2.논문제목: 발표논문 / 해양생물자원으로서 해조류 : 생물활성물질의 정제와 분자적 응용
3.저자: 홍용기(Yong Ki Hong)
4.소속기관: 한국조류학회


1.번호: 2
2.논문제목: 수거된 해양폐기물 자원화 기술 개발(Ⅰ)
3.저자: 길상인(Sang-In Keel),윤진한(Jin-Han Yun),최연석(Yeon-Seok Choi),강창구(Chang-Gu Kang),유정석(Jeong-Seok Yu)
4.소속기관: 한국해양환경·에너지학회


1.번호: 3
2.논문제목: 조건부 가치측정법을 이용한 공해상 해양생명자원 확보의 경제적 가치 추정
3.저자: 진세준,권영주,최은철
4.소속기관: 해양환경안전학회


1.번호: 4
2.논문제목: 해양수산생명자원의 국내 접근 절차 관련 법제도의 문제점 및 개선방향 : 해양수산생명자원법과 유전자원법의 비교를 중심으로
3.저자: 최석문,박수진
4.소속기관: 해양환경안전학회


1.번호: 5
2.논문제목: 공해상 해양생명자원에 대한 기업의 산업적 활용 의향에 관한 연구
3.저자: 장덕희,김윤재,진세준
4.소속기관: 해양환경안전학회


1.번호: 6
2.논문제목: 동아시아태평양 주요국가의 해양관리시스템 분석 : 해양질서관리와 해양자원관리를 중심으로
3.저자: 주종광
4.소속기관: 한국해양경찰학회


1.번호: 7
2.논문제목: 21세기 신성장동력원 해양유전자원의 개발
3.저자: 박수진
4.소속기관: 한국해양수산개발원


1.번호: 8
2.논문제목: 중국 해양자원개발의 해양강국 전략에 대한 함의
3.저자: 주현희(Hyun-Hee Ju)
4.소속기관: 한국해양비즈니스학회


1.번호: 9
2.논문제목: 우리나라 관할해역 내 해양자원 관리현황 및 정책 개선방향
3.저자: 박수진(Su Jin Park)
4.소속기관: 한국해양환경·에너지학

ValueError: Invalid extension for engine '<property object at 0x000001C94FB7DB70>': 'txt'

In [ ]:
year2 = []
degree2 = []
url2 = []


df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2
